In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# SoH + RUL for an **Euler** vehicle — reported-SoH path

Euler's feed is fundamentally different from Mahindra's, so the **method changes** (see
`config.SOH_METHOD`):

| | Mahindra (intellicar) | **Euler** |
|---|---|---|
| current | yes -> **coulomb counting** | **none** |
| voltage | yes | none (NULL) |
| reported SoH | no | **yes (`batterySoh`)** |
| SoH method | computed (coulomb) | **reported BMS SoH** |
| forecast model | condition-aware **XGBoost** (current-derived stress features) | **univariate trend** (those features don't exist) |

So for Euler we **cannot** run the Mahindra XGBoost model — its 20 features are all derived from
current/voltage. Instead we read the BMS's own `batterySoh`, clean it, and forecast the trend.

**Caveats found in the Euler data** (scanned 8 vehicles): reported SoH is *coarse* (~1-1.5 pp steps)
and *noisy upward* (BMS recalibrations), and some vehicles show **battery-replacement events** (SoH
resets up while the odometer jumps). We handle both: cut at a replacement, then take the monotonic
(non-increasing) envelope — the same "SoH can't truly heal" rule used for Mahindra.

In [ ]:
import numpy as np, pandas as pd, json, re
import matplotlib.pyplot as plt
import config

VIN = 'MD9EMHDL22E217062'          # clean steady decline, no pack swap, currently ~80% EOL
fp = Path(f'data/euler/feed/{VIN}.parquet')

# --- monthly-sample import from S3 (skips if already cached) ---
if not fp.exists():
    from dotenv import load_dotenv; load_dotenv('.env')
    import boto3; from botocore.config import Config
    from concurrent.futures import ThreadPoolExecutor, as_completed
    B = os.environ['S3_BUCKET']
    s3 = boto3.client('s3', config=Config(max_pool_connections=48, retries={'max_attempts':5,'mode':'adaptive'}))
    PREFIX = config.OEM_FEEDS['euler']['prefix']; COLS = ['eventAt','vin','batterySoc','batterySoh','odometer','batteryTemperature']
    EXPR = f"SELECT {', '.join('s.'+c for c in COLS)} FROM s3object s WHERE s.vin='{VIN}'"
    kids = lambda p: sorted(x['Prefix'] for x in s3.list_objects_v2(Bucket=B,Prefix=p,Delimiter='/').get('CommonPrefixes',[]))
    dom  = lambda d: int(re.search(r'day=(\d{2})',d).group(1))
    days = [min(kids(mo),key=lambda d:abs(dom(d)-15)) for y in kids(PREFIX) for mo in kids(y) if kids(mo)]
    def sel(k):
        r=s3.select_object_content(Bucket=B,Key=k,ExpressionType='SQL',Expression=EXPR,
            InputSerialization={'Parquet':{}},OutputSerialization={'JSON':{'RecordDelimiter':chr(10)}})
        buf=bytearray()
        for ev in r['Payload']:
            if 'Records' in ev: buf+=ev['Records']['Payload']
        return [json.loads(l) for l in buf.decode().splitlines() if l.strip()]
    rows=[]
    for day in days:
        keys=[o['Key'] for pg in s3.get_paginator('list_objects_v2').paginate(Bucket=B,Prefix=day) for o in pg.get('Contents',[]) if o['Key'].endswith('.parquet')][:60]
        with ThreadPoolExecutor(max_workers=48) as pool:
            for f in as_completed([pool.submit(sel,k) for k in keys]):
                try: rows+=f.result()
                except Exception: pass
    df=pd.DataFrame(rows); df['t']=pd.to_datetime(df['eventAt'].astype('int64'),unit='ms')
    fp.parent.mkdir(parents=True,exist_ok=True); df.to_parquet(fp,index=False)

df = pd.read_parquet(fp)
df['t'] = pd.to_datetime(df['t']) if 't' in df else pd.to_datetime(df['eventAt'].astype('int64'),unit='ms')
for c in ['batterySoc','batterySoh','odometer','batteryTemperature']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['month'] = df['t'].dt.to_period('M').dt.to_timestamp()
print(f'{VIN}: {len(df):,} sampled rows, {df.month.nunique()} months, {df.t.min().date()}..{df.t.max().date()}')

## 1. Reported SoH -> clean monthly series

Monthly median of `batterySoh` (drop 0/garbage), cut at any battery-replacement event
(SoH jumps up >4 pp **and** odometer jumps), then take the monotonic non-increasing envelope.

In [ ]:
g = df[(df['batterySoh']>0)&(df['batterySoh']<=100)].groupby('month')
s = pd.DataFrame({'soh_rep': g['batterySoh'].median(), 'odo': g['odometer'].max(),
                  'temp': g['batteryTemperature'].mean()}).reset_index()
# detect first battery-replacement / reset: SoH up >4pp AND odometer jump >4000 km
swap = (s['soh_rep'].diff()>4) & (s['odo'].diff()>4000)
cut = swap.idxmax() if swap.any() else len(s)
seg = s.iloc[:cut].copy() if swap.any() else s.copy()
seg['soh_mono'] = seg['soh_rep'].cummin()                     # monotonic envelope (no BMS heal-ups)
seg['age_yr'] = (seg['month'] - seg['month'].iloc[0]).dt.days/365.25
print(f"kept {len(seg)} months (cut a replacement event: {bool(swap.any())})")
print(f"reported SoH {seg['soh_rep'].iloc[0]:.1f}% -> {seg['soh_rep'].iloc[-1]:.1f}%   "
      f"monotonic {seg['soh_mono'].iloc[0]:.1f}% -> {seg['soh_mono'].iloc[-1]:.1f}%   over {seg['age_yr'].iloc[-1]:.1f} yr")

In [ ]:
fig, ax = plt.subplots(figsize=(11,5.5))
ax.plot(s['month'], s['soh_rep'], 'o', color='0.7', ms=4, label='reported batterySoh (monthly median, raw)')
ax.plot(seg['month'], seg['soh_mono'], 'o-', color='C0', ms=4, lw=1.6, label='monotonic envelope (used for forecast)')
if (s['soh_rep'].diff()>4).any() and ((s['soh_rep'].diff()>4)&(s['odo'].diff()>4000)).any():
    ev = s['month'].iloc[(( s['soh_rep'].diff()>4)&(s['odo'].diff()>4000)).idxmax()]
    ax.axvline(ev, ls='--', color='purple', lw=1.2, label='battery replacement (cut)')
ax.axhline(80, ls=':', color='red', lw=1); ax.text(s['month'].iloc[0], 80.6, '80% EOL', color='red', fontsize=9)
ax.set_ylabel('SoH (%)'); ax.set_title(f'Euler {VIN} — reported SoH (BMS) vs cleaned envelope'); ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 2. Forecast (univariate degradation trend)

No current/voltage -> no condition-aware features, so we fit a **square-root-of-time** fade curve
(`SoH = S0 - k*sqrt(age)`, the physical calendar-fade shape) to the monotonic envelope and roll it
forward. A linear fit is shown for comparison. We read off **RUL = time until 80% SoH** and the
projected SoH at the 5-year Euler warranty.

In [ ]:
age = seg['age_yr'].to_numpy(); soh = seg['soh_mono'].to_numpy()
# sqrt-time fit: soh = a + b*sqrt(age)
a, b = np.polyfit(np.sqrt(age), soh, 1)[::-1]
c, d = np.polyfit(age, soh, 1)[::-1]                    # linear: soh = c + d*age
fyr = np.linspace(0, 7, 200)
sqrt_fc = a + b*np.sqrt(fyr); lin_fc = c + d*fyr
def cross80_sqrt():
    r = (80 - a)/b
    return r*r if r>0 else np.inf
eol_sqrt = cross80_sqrt(); eol_lin = (80 - c)/d
wyr = config.warranty_for('euler','')[0]               # default Euler warranty (years)
soh_at_warr = a + b*np.sqrt(wyr)
age_now = age[-1]
print(f"warranty (Euler default): {wyr} yr,  {config.warranty_for('euler','')[1]:,} km")
print(f"current age {age_now:.1f} yr, current SoH {soh[-1]:.1f}%")
print(f"projected EOL (80%):  sqrt-fit {eol_sqrt:.1f} yr   |   linear {eol_lin:.1f} yr")
print(f"RUL to 80% (sqrt-fit): {max(eol_sqrt-age_now,0):.1f} yr  ({max((eol_sqrt-age_now)*12,0):.0f} months) from now")
print(f"projected SoH at {wyr}-yr warranty: {soh_at_warr:.1f}%  -> {'AT-RISK (<80%)' if soh_at_warr<80 else 'OK'}")

fig, ax = plt.subplots(figsize=(11,6))
ax.plot(age, soh, 'o-', color='0.5', ms=5, lw=1.6, label='actual (monotonic reported SoH)')
fcol = 'tab:red' if soh_at_warr<80 else 'tab:green'
fut = fyr>=age_now
ax.plot(fyr[fut], sqrt_fc[fut], '--', color=fcol, lw=2.2, label='forecast — sqrt-time fade')
ax.plot(fyr[fut], lin_fc[fut], ':', color=fcol, lw=1.4, alpha=0.7, label='forecast — linear (compare)')
ax.axhline(80, ls=':', color='red', lw=1.1); ax.text(0.05, 80.5, '80% EOL', color='red', fontsize=9)
ax.axvline(wyr, ls='-.', color='green', lw=1.3); ax.text(wyr-0.05, ax.get_ylim()[0]+1, f'{wyr}-yr warranty', rotation=90, fontsize=8, ha='right', color='green')
if np.isfinite(eol_sqrt): ax.plot([eol_sqrt],[80],'v',color=fcol,ms=10)
ax.set_xlabel('age since first telemetry (years)'); ax.set_ylabel('SoH (%)')
ax.set_title(f'Euler {VIN} — reported SoH + univariate forecast  (RUL to 80%: ~{max((eol_sqrt-age_now)*12,0):.0f} months)')
ax.set_xlim(0, 7); ax.grid(alpha=0.3); ax.legend(fontsize=9, loc='lower left')
plt.tight_layout(); plt.show()